# Unix Shell 1

[ACME lab](https://labs.acme.byu.edu/DataScienceEssentials/UnixShell1/UnixShell1.html) · Volume 1 · Vikunja task #46

This notebook works through all six problems using the official starter archive and an accompanying Bash script. Run it from this notebook’s directory. It downloads the starter files, executes the script in a temporary directory, and retains the final script and compressed result in `UnixShell1/`.

## Navigation and command help

`pwd` reports the working directory, `ls` lists its entries, and `cd` changes location. `cd ..` moves to the parent directory; `cd ~` goes home. Paths after `cd Shell1` are relative to that directory. Use `command --help` or `man command` to inspect flags. For `ls`, `-a` includes hidden entries, `-l` gives details, and `-S` sorts by decreasing size.

**Problems 1–2.** The script enters `Shell1`, uses `printf` to report its path, and runs `ls -alS` so all three listing requirements hold in one command.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile
import subprocess
import shutil
import tarfile

lab = Path("UnixShell1")
workspace = TemporaryDirectory(prefix="acme-shell1-")
work = Path(workspace.name)
url = "https://labs.acme.byu.edu/_downloads/e9ca7e2c36ec1001a8d83d1f659f7672/UnixShell1.zip"
urlretrieve(url, work / "starter.zip")
with ZipFile(work / "starter.zip") as outer:
    (work / "Shell1.zip").write_bytes(outer.read("Shell1.zip"))
with ZipFile(work / "Shell1.zip") as inner:
    names = inner.namelist()
shutil.copy2(lab / "unixshell1.sh", work / "unixshell1.sh")
print(f"Downloaded official starter archive: {len(names)} entries")
print("Lab workspace:", work)
print("Script:", lab / "unixshell1.sh")

Downloaded official starter archive: 95 entries
Lab workspace: /tmp/acme-shell1-m2m9hgla
Script: UnixShell1/unixshell1.sh


## File and directory operations

`mkdir` creates a directory; `rm -r` removes a directory tree; `mv` moves or renames an entry. Shell wildcards match groups of filenames: `*.jpg`, for example, matches JPEG files in the current directory. These patterns do not search nested directories.

**Problems 3–4.** The script removes `Audio/`, creates `Documents/`, `Photos/`, and `Python/`, renames `Random/` to `Files/`, and moves the top-level JPEG, text, and Python files into their new directories.

## Search and permissions

`find` searches filenames in a directory tree. `grep -nr` searches file contents recursively and shows line numbers. `chmod u+x` gives the owner execute permission; `./script.sh` executes a script from a relative path.

**Problem 5.** `find` locates `organize_photos.sh` inside the renamed `Files/` tree. The script moves it into `Scripts/`, adds owner execute permission, then runs it to collect nested JPEGs.

In [2]:
print((lab / "unixshell1.sh").read_text())

#!/usr/bin/env bash
set -euo pipefail

# Run from the directory containing Shell1.zip.
rm -rf Shell1
unzip -q Shell1.zip

# Problem 1: navigate and report the current directory without echo.
cd Shell1
printf 'Working in %s\n' "$PWD"

# Problem 2: include hidden entries, use long format, and sort by size.
ls -alS

# Problem 3: remove, create, and rename directories.
rm -r Audio
mkdir Documents Photos Python
mv Random Files

# Problem 4: organize top-level files by extension.
mv ./*.jpg Photos/
mv ./*.txt Documents/
mv ./*.py Python/

# Problem 5: find, move, permit, and execute the photo organizer.
organizer=$(find Files -name organize_photos.sh -type f -print -quit)
printf 'Found photo organizer: %s\n' "$organizer"
mv "$organizer" Scripts/
chmod u+x Scripts/organize_photos.sh
bash Scripts/organize_photos.sh

cd ..
tar -czf UnixShell1.tar.gz Shell1
printf 'Created %s/UnixShell1.tar.gz\n' "$PWD"



In [3]:
result = subprocess.run(["bash", "unixshell1.sh"], cwd=work, text=True, capture_output=True, check=True)
print(result.stdout)
if result.stderr:
    print("stderr:", result.stderr)
shutil.copy2(work / "UnixShell1.tar.gz", lab / "UnixShell1.tar.gz")
print("Saved archive:", lab / "UnixShell1.tar.gz")

Working in /tmp/acme-shell1-m2m9hgla/Shell1
total 344
-rw-r--r-- 1 andrew dev 81167 Aug 20  2019 img_1987.jpg
-rw-r--r-- 1 andrew dev 72530 Aug 20  2019 img_1842.jpg
-rw-r--r-- 1 andrew dev 51193 Aug 20  2019 img_1796.jpg
-rw-r--r-- 1 andrew dev 35242 Aug 20  2019 img_1879.jpg
-rw-r--r-- 1 andrew dev  7996 Aug 20  2019 files.txt
-rw-r--r-- 1 andrew dev  7536 Aug 20  2019 newfiles.txt
-rw-r--r-- 1 andrew dev  6148 Aug 20  2019 .DS_Store
drwx------ 6 andrew dev  4096 Aug 20  2019 .
drwx------ 3 andrew dev  4096 Nov 27 08:47 ..
drwx------ 3 andrew dev  4096 Aug 20  2019 Audio
drwx------ 9 andrew dev  4096 Aug 20  2019 Random
drwx------ 2 andrew dev  4096 Aug 20  2019 Scripts
drwx------ 2 andrew dev  4096 Aug 20  2019 Test
-rw-r--r-- 1 andrew dev  1027 Aug 20  2019 review.txt
-rw-r--r-- 1 andrew dev   881 Aug 20  2019 words.txt
-rw-r--r-- 1 andrew dev   721 Aug 20  2019 project.py
-rw-r--r-- 1 andrew dev   612 Aug 20  2019 doc3.txt
-rw-r--r-- 1 andrew dev   427 Aug 20  2019 doc1.txt
-rw-r-

## Check the result

The checks below verify each requested operation against the extracted starter data. The compressed archive is read back independently, including its hidden file. The same script can be run again from a folder containing the official `Shell1.zip` file.

In [4]:
tree = work / "Shell1"
assert tree.is_dir() and not (tree / "Audio").exists()
assert (tree / "Files").is_dir() and not (tree / "Random").exists()
for folder in ("Documents", "Photos", "Python", "Scripts"):
    assert (tree / folder).is_dir()
for pattern, folder in (("*.jpg", "Photos"), ("*.txt", "Documents"), ("*.py", "Python")):
    assert not list(tree.glob(pattern)), f"Unmoved top-level {pattern}"
    print(f"{folder}: {len(list((tree / folder).glob(pattern)))} {pattern} files")
organizer = tree / "Scripts" / "organize_photos.sh"
assert organizer.exists() and organizer.stat().st_mode & 0o100
assert not list((tree / "Files").rglob("*.jpg"))
print("Organizer executable:", bool(organizer.stat().st_mode & 0o100))
with tarfile.open(lab / "UnixShell1.tar.gz", "r:gz") as archive:
    archived = archive.getnames()
assert "Shell1/.DS_Store" in archived
assert "Shell1/Scripts/organize_photos.sh" in archived
print("Archive entries:", len(archived))
print("Checks passed: directories, file moves, photo organizer, permissions, archive")

Photos: 14 *.jpg files
Documents: 12 *.txt files
Python: 4 *.py files
Organizer executable: True
Archive entries: 77
Checks passed: directories, file moves, photo organizer, permissions, archive


In [5]:
matches = sorted(str(p.relative_to(tree)) for p in tree.rglob("organize_photos.sh"))
print("find result:", matches)
people = tree / "Documents" / "people.txt"
for number, line in enumerate(people.read_text().splitlines(), 1):
    if "Mary" in line:
        print(f"grep -nr result: Documents/people.txt:{number}:{line}")
print("Script permissions:", oct(organizer.stat().st_mode & 0o777))

find result: ['Scripts/organize_photos.sh']
grep -nr result: Documents/people.txt:2:female,Mary,31
Script permissions: 0o700


## Git and remote access

**Problem 6.** The deliverables in the lab folder are `unixshell1.sh` and `UnixShell1.tar.gz`; they are committed and pushed with this notebook. A usual Git sequence is `git status`, `git add <files>`, `git commit -m <message>`, and `git push`. The notebook lives beside the lab folder so no extra file is submitted inside `UnixShell1/`.

For remote systems, `ssh user@host` starts an encrypted login session, while `scp file user@host:path/` transfers a file. Use `scp -r` for directory trees. No remote login is needed for this lab.